In [ ]:
# Check if a GPU is available
!nvidia-smi || echo 'No GPU available. You may be on a CPU-only runtime.'

Thu Dec 18 09:37:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models


In [ ]:
import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

Ollama server started as background process (PID: 1716 )
Waited 5 seconds for the server to initialize.


In [ ]:
!ollama pull llama3:8b

In [ ]:
!pip install -r '/content/drive/MyDrive/RAG/requirements.txt'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.2/328.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from pathlib import Path
import sys
import time
from pprint import pprint

# Add src to path
sys.path.append(str(Path('/content/drive/MyDrive/RAG/src').parent))

from src.config.config import Config
from src.document_ingestion.document_processor import DocumentProcessor
from src.vectorstore.vectorstore import VectorStore
from src.graph_builder.graph_builder import GraphBuilder

In [ ]:
llm = Config.get_llm()
embedding = Config.get_embedding_model()
doc_processor = DocumentProcessor(embedding)
vector_store = VectorStore(embedding)

pdfs = '/content/drive/MyDrive/RAG/datasets'

document_processor = DocumentProcessor(embedding)
documents = document_processor.process(pdfs)

# Create vector store
vector_store.create_vectorstore(documents)

/content/drive/MyDrive/RAG/src/config/config.py:18: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  return Ollama(model=model_name)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma vector store created and persisted to /content/drive/MyDrive/RAG/chroma_db


In [ ]:
# Build graph
graph_builder = GraphBuilder(
    retriever=vector_store.get_retriever(),
    llm=llm
)
builder = graph_builder.build() # Assign the returned workflow to a variable

In [ ]:
import pandas as pd

dataset = pd.read_excel('/content/drive/MyDrive/RAG/datasets/datasets.xlsx', sheet_name="final")
# dataset.rename(columns={'answer': 'answer_ref'}, inplace=True)

In [ ]:
dataset[70:75]

,index,dataset_name,type,doc,question,answer_ref,passage_1_text,passage_1_page,passage_2_text,passage_2_page,Unnamed: 10,Unnamed: 11
70,77.0,dataset_147,human,Z:/AI_Projects/RAG_R_and_D_Project/src/Austral...,What percent of the Australian year 3 students...,"88,7 percent","In 2019, the proportion of Year 3 students at ...",11,NaN,NaN,NaN,NaN
71,78.0,dataset_148,human,Z:/AI_Projects/RAG_R_and_D_Project/src/Austral...,How many Australian parents were satisfied wit...,88 percent (possible additional: including 48 ...,"In November 2020, a poll of Australians ages 1...",12,NaN,NaN,NaN,NaN
72,79.0,dataset_149,human,Z:/AI_Projects/RAG_R_and_D_Project/src/Austria...,When did PISA results again confirmed a lack o...,December 2019,"In December 2019, PISA results again confirmed...",13,NaN,NaN,NaN,NaN
73,80.0,dataset_150,human,Z:/AI_Projects/RAG_R_and_D_Project/src/Austria...,What measure adddressed the increased need for...,Expansion \nof school-based support of German ...,This has been addressed by the expansion \nof ...,8,NaN,NaN,NaN,NaN
74,81.0,dataset_151,human,Z:/AI_Projects/RAG_R_and_D_Project/src/Bahrain...,Which article of the Bahrani Educational Evalu...,Article 4,Article 4 of the Educational Evaluation Law (t...,6,NaN,NaN,NaN,NaN


In [ ]:
%%time

import os
import math

answers = {}

for i, q in enumerate(dataset['question'].tolist()):
    # Skip NaN values in the question list
    if isinstance(q, float) and math.isnan(q):
        print(f"Skipping question at index {i} due to NaN value.")
        continue

    response = graph_builder.run(q)
    answers[i] = {} # Initialize the dictionary for the current query index
    answers[i]['query'] = response['question']
    answers[i]['answer'] = response['answer']
    answers[i]['retrieved_docs'] = [] # Initialize retrieved_docs as a list of dictionaries

    for doc in response['retrieved_docs']:
        answers[i]['retrieved_docs'].append({
            'content': doc.page_content,
            'doc_name': doc.metadata.get('title', os.path.basename(doc.metadata['source'])),
            'page': doc.metadata['page'],
            'page_label': doc.metadata['page_label'],
            'source': doc.metadata['source']
        })


Skipping question at index 170 due to NaN value.
CPU times: user 12.1 s, sys: 1.88 s, total: 14 s
Wall time: 36min 57s


In [ ]:
import pandas as pd

data_for_df = []
for idx, item in answers.items():
    base_row = {
        'index': idx,
        'query': item['query'],
        'answer': item['answer']
    }

    # Check if 'retrieved_docs' is a list and not empty
    if 'retrieved_docs' in item and isinstance(item['retrieved_docs'], list) and item['retrieved_docs']:
        for doc_info in item['retrieved_docs']:
            row = base_row.copy()
            row.update({
                'content': doc_info.get('content'),
                'doc_name': doc_info.get('doc_name'),
                'page': doc_info.get('page'),
                'page_label': doc_info.get('page_label'),
                'source': doc_info.get('source')
            })
            data_for_df.append(row)
    else:
        # Handle cases where retrieved_docs might be missing, not a list, or empty
        row = base_row.copy()
        row.update({
            'content': None,
            'doc_name': None,
            'page': None,
            'page_label': None,
            'source': None
        })
        data_for_df.append(row)

df_answers = pd.DataFrame(data_for_df)
display(df_answers.head())

,index,query,answer,content,doc_name,page,page_label,source
0,0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
1,0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
2,0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
3,0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
4,1,"How many public, private, and charter schools ...",Let's walk through my thought process step by ...,There are 218 public schools with a student po...,Abu Dhabi.pdf,0,1,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...


In [ ]:
merged_df_by_qa = pd.merge(dataset, df_answers, left_on='question', right_on='query', how='left')
display(merged_df_by_qa.head())

,index_x,dataset_name,type,doc,question,answer_ref,passage_1_text,passage_1_page,passage_2_text,passage_2_page,Unnamed: 10,Unnamed: 11,index_y,query,answer,content,doc_name,page,page_label,source
0,1.0,dataset_1,synthetic,"""Z:\AI_Projects\RAG_R_and_D_Project\src\Abu Dh...",Which month is set aside each year in the UAE ...,March,Building on the Abu Dhabi Reads campaign launc...,3,"Every March, the UAE celebrates a month of rea...",6,NaN,NaN,0.0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2.0,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
1,1.0,dataset_1,synthetic,"""Z:\AI_Projects\RAG_R_and_D_Project\src\Abu Dh...",Which month is set aside each year in the UAE ...,March,Building on the Abu Dhabi Reads campaign launc...,3,"Every March, the UAE celebrates a month of rea...",6,NaN,NaN,0.0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2.0,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
2,1.0,dataset_1,synthetic,"""Z:\AI_Projects\RAG_R_and_D_Project\src\Abu Dh...",Which month is set aside each year in the UAE ...,March,Building on the Abu Dhabi Reads campaign launc...,3,"Every March, the UAE celebrates a month of rea...",6,NaN,NaN,0.0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2.0,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
3,1.0,dataset_1,synthetic,"""Z:\AI_Projects\RAG_R_and_D_Project\src\Abu Dh...",Which month is set aside each year in the UAE ...,March,Building on the Abu Dhabi Reads campaign launc...,3,"Every March, the UAE celebrates a month of rea...",6,NaN,NaN,0.0,Which month is set aside each year in the UAE ...,I'll walk you through my thought process step ...,international reading performance benchmark fo...,Abu Dhabi.pdf,2.0,3,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...
4,2.0,dataset_2,synthetic,"""Z:\AI_Projects\RAG_R_and_D_Project\src\Abu Dh...","How many public, private, and charter schools ...","Abu Dhabi has 218 public schools, 203 private ...",There are 218 public schools with a student po...,1,"Exhibit 1: Demographics for Abu Dhabi Schools,...",2,NaN,NaN,1.0,"How many public, private, and charter schools ...",Let's walk through my thought process step by ...,There are 218 public schools with a student po...,Abu Dhabi.pdf,0.0,1,/content/drive/MyDrive/RAG/datasets/Abu Dhabi ...


In [ ]:
merged_df_by_qa.to_csv('/content/drive/MyDrive/RAG/datasets/rag_with_llama_result.csv', index=False)